In [38]:
import sqlite3
import pandas as pd
import os
import sys
import subprocess
import kagglehub
import pyarrow.parquet as pq
import fastparquet

# Connect to the SQLite database
conn = sqlite3.connect('Data/nba.db')

# NBA dataset from Kaggle
nba_path = "https://www.kaggle.com/datasets/eoinamoore/historical-nba-data-and-player-box-scores/data"

In [4]:
print(dir(kagglehub))

['KaggleDatasetAdapter', 'PolarsFrameType', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'auth', 'cache', 'clients', 'colab_cache_resolver', 'competition', 'competition_download', 'config', 'dataset_download', 'dataset_load', 'dataset_upload', 'datasets', 'datasets_enums', 'datasets_helpers', 'enum', 'env', 'exceptions', 'gcs_upload', 'get_package_asset_path', 'handle', 'http_resolver', 'integrity', 'kaggle_cache_resolver', 'kagglehub', 'load_dataset', 'logger', 'login', 'model_download', 'model_upload', 'models', 'models_helpers', 'notebook_output_download', 'notebooks', 'package_import', 'packages', 'registry', 'resolver', 'signing', 'tracker', 'utility_script_install', 'utility_scripts', 'whoami']


In [14]:
os.listdir(dataset_path)

['Games.csv',
 'LeagueSchedule24_25.csv',
 'LeagueSchedule25_26.csv',
 'PlayByPlay.parquet',
 'Players.csv',
 'PlayerStatistics.csv',
 'PlayerStatisticsAdvanced.csv',
 'PlayerStatisticsMisc.csv',
 'PlayerStatisticsScoring.csv',
 'PlayerStatisticsUsage.csv',
 'TeamHistories.csv',
 'TeamStatistics.csv',
 'TeamStatisticsAdvanced.csv',
 'TeamStatisticsFourFactors.csv',
 'TeamStatisticsMisc.csv',
 'TeamStatisticsScoring.csv']

### Players table ingestion

In [ ]:
# Read the Players CSV file

dataset_path = kagglehub.dataset_download("eoinamoore/historical-nba-data-and-player-box-scores")
players_df = pd.read_csv(os.path.join(dataset_path, "Players.csv"))

# Ingest the data into the players table
players_df.to_sql('players', conn, if_exists='replace', index=False)

# Commit and close the connection
conn.commit()

print(f"Successfully ingested {len(players_df)} records into the players table.")
print(f"\nColumns: {players_df.columns.tolist()}")

conn.close()

### Games table ingestion

In [16]:
games_df = pd.read_csv(os.path.join(dataset_path, "Games.csv"))
# Ingest the data into the games table
games_df.to_sql('games', conn, if_exists='replace', index=False)
# Commit and close the connection
conn.commit()
print(f"Successfully ingested {len(games_df)} records into the games table.")
print(f"\nColumns: {games_df.columns.tolist()}")

C:\Users\Mark Rozenberg\AppData\Local\Temp\ipykernel_24752\760540420.py:1: DtypeWarning: Columns (0: gameSubtype, 1: gameSubLabel, 2: arenaName, 3: arenaCity, 4: arenaState, 5: officials) have mixed types. Specify dtype option on import or set low_memory=False.
  games_df = pd.read_csv(os.path.join(dataset_path, "Games.csv"))


Successfully ingested 73119 records into the games table.

Columns: ['gameId', 'gameDateTimeEst', 'hometeamCity', 'hometeamName', 'hometeamId', 'awayteamCity', 'awayteamName', 'awayteamId', 'homeScore', 'awayScore', 'winner', 'gameType', 'gameSubtype', 'gameLabel', 'gameSubLabel', 'seriesGameNumber', 'attendance', 'arenaId', 'arenaName', 'arenaCity', 'arenaState', 'officials']


### PlayByPlay table ingestion

In [ ]:
plays_df = pd.read_parquet(os.path.join(dataset_path, "PlayByPlay.parquet"), engine='fastparquet')
# Ingest the data into the games table
chunk_size = 50_000
# plays_df.to_sql('plays', conn, if_exists='replace', index=False, chunksize=10000)

# Commit and close the connection
conn.commit()
print(f"Successfully ingested {len(plays_df)} records into the plays table.")
print(f"\nColumns: {plays_df.columns.tolist()}")

In [ ]:
parquet_path = os.path.join(dataset_path, "PlayByPlay.parquet")
pf = pq.ParquetFile(parquet_path)

first = True
total = 0

for i, batch in enumerate(pf.iter_batches(batch_size=50_000), start=1):
    chunk = batch.to_pandas()

    # Optional: reduce memory footprint
    # for c in chunk.select_dtypes(include=["float64"]).columns:
    #     chunk[c] = pd.to_numeric(chunk[c], downcast="float")

    chunk.to_sql(
        "plays",
        conn,
        if_exists="replace" if first else "append",
        index=False,
        chunksize=5_000,   # SQL insert chunk size
        method=None        # safer memory profile than large "multi" statements
    )

    first = False
    total += len(chunk)
    print(f"plays: wrote chunk {i}, rows={len(chunk)}")

conn.commit()
print(f"Successfully ingested {total} records into the plays table.")
print(f"\nColumns: {chunk.columns.tolist()}")

In [39]:
conn.execute("drop table if exists plays")
conn.execute("drop table if exists players")
conn.execute("drop table if exists games")
conn.commit()

OperationalError: database is locked

In [27]:
# List all tables in the nba.db database
tables = conn.execute(
    "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name"
).fetchall()

print("Tables in nba.db:")
for table in tables:
    print(f"  - {table[0]}")

    # Get row counts for each table
    for table_name in [t[0] for t in tables]:
        count = conn.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
        print(f"  {table_name}: {count:,} rows")

    # Get schema info for each table
    print("\nTable schemas:")
    for table_name in [t[0] for t in tables]:
        columns = conn.execute(f"PRAGMA table_info({table_name})").fetchall()
        print(f"\n{table_name}:")
        for col in columns:
            print(f"  - {col[1]}: {col[2]}")

Tables in nba.db:
  - games
  - players


In [30]:
conn.execute("pragma table_info(players)").fetchall()

[(0, 'personId', 'INTEGER', 0, None, 0),
 (1, 'firstName', 'TEXT', 0, None, 0),
 (2, 'lastName', 'TEXT', 0, None, 0),
 (3, 'birthDate', 'REAL', 0, None, 0),
 (4, 'school', 'TEXT', 0, None, 0),
 (5, 'country', 'TEXT', 0, None, 0),
 (6, 'heightInches', 'REAL', 0, None, 0),
 (7, 'bodyWeightLbs', 'REAL', 0, None, 0),
 (8, 'guard', 'TEXT', 0, None, 0),
 (9, 'forward', 'TEXT', 0, None, 0),
 (10, 'center', 'TEXT', 0, None, 0),
 (11, 'draftYear', 'REAL', 0, None, 0),
 (12, 'draftRound', 'REAL', 0, None, 0),
 (13, 'draftNumber', 'REAL', 0, None, 0)]

In [36]:
print('length players: ', len(pd.read_sql("SELECT * FROM players", conn)))
print('length games: ', len(pd.read_sql("SELECT * FROM games", conn)))

length players:  6691
length games:  73119


In [37]:
conn.commit()
conn.close()